# PostgresManager Usage Example

Demonstrates how to use the PostgresManager class for common database operations.

**Note**: Make sure to update the connection parameters (host, port, database, user, password) before running.

In [ ]:
from egghouse.database import PostgresManager
import logging

# Setup logging to see query execution
logging.basicConfig(level=logging.INFO)

## 1. Connecting to PostgreSQL

In [ ]:
# Initialize connection
db = PostgresManager(
    host='localhost',
    port=5432,
    database='test_db',  # Make sure this database exists
    user='your_user',
    password='your_password',
    log_queries=True
)

# Alternative: Using context manager (auto-close)
# with PostgresManager(host='localhost', database='test_db', ...) as db:
#     db.create_table(...)

## 2. Database Operations

In [ ]:
# List databases
databases = db.list_databases()
print(f"Available databases: {len(databases)}")
for db_info in databases[:5]:  # Show first 5
    print(f"  - {db_info['name']}: {db_info['size']}")

## 3. Schema Operations

In [ ]:
# Create schema
db.create_schema('research')

# List schemas
schemas = db.list_schemas()
print(f"Available schemas: {[s['name'] for s in schemas]}")

## 4. Table Operations

In [ ]:
# Create table
db.create_table(
    'solar_events',
    {
        'id': 'SERIAL PRIMARY KEY',
        'event_type': 'VARCHAR(50) NOT NULL',
        'start_time': 'TIMESTAMP NOT NULL',
        'end_time': 'TIMESTAMP',
        'intensity': 'FLOAT',
        'description': 'TEXT',
        'created_at': 'TIMESTAMP DEFAULT NOW()'
    },
    schema='research'
)

# Check if table exists
exists = db.table_exists('solar_events', schema='research')
print(f"Table 'research.solar_events' exists: {exists}")

In [ ]:
# Describe table
columns = db.describe_table('solar_events', schema='research')
print("Table structure:")
for col in columns:
    print(f"  - {col['name']}: {col['type']} "
          f"(nullable: {col['is_nullable']}, default: {col['default_value']})")

In [ ]:
# List tables
tables = db.list_tables(schema='research')
print(f"Tables in 'research' schema:")
for table in tables:
    print(f"  - {table['name']}: {table['size']}")

## 5. Data Operations

### INSERT

In [ ]:
# Insert single record
db.insert(
    'solar_events',
    {
        'event_type': 'solar_flare',
        'start_time': '2025-01-15 10:30:00',
        'intensity': 8.5,
        'description': 'X-class flare observed'
    },
    schema='research'
)

# Insert multiple records
db.insert(
    'solar_events',
    [
        {
            'event_type': 'CME',
            'start_time': '2025-01-16 14:20:00',
            'intensity': 7.2,
            'description': 'Coronal mass ejection'
        },
        {
            'event_type': 'solar_wind',
            'start_time': '2025-01-17 08:15:00',
            'intensity': 5.8,
            'description': 'High-speed solar wind stream'
        }
    ],
    schema='research'
)
print("Records inserted.")

### SELECT

In [ ]:
# Select all data
all_events = db.select('solar_events', schema='research')
print(f"Total events: {len(all_events)}")
for event in all_events:
    print(f"  - {event['event_type']}: {event['start_time']} (intensity: {event['intensity']})")

In [ ]:
# Select with WHERE clause
flares = db.select(
    'solar_events',
    where={'event_type': 'solar_flare'},
    schema='research'
)
print(f"Solar flares: {len(flares)}")

In [ ]:
# Select with columns and ordering
recent_events = db.select(
    'solar_events',
    columns=['event_type', 'start_time', 'intensity'],
    order_by='intensity DESC',
    limit=2,
    schema='research'
)
print(f"Top 2 events by intensity:")
for event in recent_events:
    print(f"  - {event['event_type']}: {event['intensity']}")

### COUNT

In [ ]:
total = db.count('solar_events', schema='research')
flare_count = db.count('solar_events', where={'event_type': 'solar_flare'}, schema='research')
print(f"Total events: {total}")
print(f"Solar flares: {flare_count}")

### UPDATE

In [ ]:
affected = db.update(
    'solar_events',
    data={'intensity': 9.0, 'description': 'Major X-class flare observed'},
    where={'event_type': 'solar_flare'},
    schema='research'
)
print(f"Updated {affected} rows")

# Verify update
updated = db.select('solar_events', where={'event_type': 'solar_flare'}, schema='research')
print(f"Updated flare intensity: {updated[0]['intensity']}")

### DELETE

In [ ]:
deleted = db.delete(
    'solar_events',
    where={'event_type': 'solar_wind'},
    schema='research'
)
print(f"Deleted {deleted} rows")

# Verify deletion
remaining = db.count('solar_events', schema='research')
print(f"Remaining events: {remaining}")

## 6. Raw SQL Execution

In [ ]:
# Execute custom query
result = db.execute(
    """
    SELECT event_type, AVG(intensity) as avg_intensity
    FROM research.solar_events
    GROUP BY event_type
    ORDER BY avg_intensity DESC
    """,
    fetch=True
)
print("Average intensity by event type:")
for row in result:
    print(f"  - {row['event_type']}: {row['avg_intensity']:.2f}")

## 7. Cleanup

In [ ]:
# Truncate table
db.truncate('solar_events', schema='research')
print("Table truncated")

# Drop table
db.drop_table('solar_events', schema='research')
print("Table dropped")

# Drop schema
db.drop_schema('research', cascade=True)
print("Schema dropped")

# Close connection
db.close()
print("Connection closed")